In [1]:
from itertools import combinations

import pandas as pd

In [2]:
#predict probability of winning
def predict_proba(A_elo, B_elo):
    return 1 / (1 + 10 ** ((B_elo - A_elo) / 400))

In [4]:
elo_teams = pd.read_csv('final_elo_team.csv')
elo_teams

,TeamID,TeamELO
0,1228,1976.446582
1,1328,1865.209005
2,1106,1235.938982
3,1354,1274.504728
4,1112,1992.047421
...,...,...
744,1478,1155.979444
745,1479,1341.454910
746,1480,1258.946093
747,3480,1351.409092


In [5]:
elo_coaches = pd.read_csv('final_elo_coach.csv')
elo_coaches

,Season,CoachName,CoachELO,TeamID
0,2025.0,fran_mccaffery,1855.833027,1234.0
1,2025.0,rick_pitino,2143.617542,1385.0
2,2025.0,leonard_hamilton,1787.864123,1199.0
3,2025.0,rick_barnes,2129.903478,1397.0
4,2025.0,kelvin_sampson,2243.025621,1222.0
...,...,...,...,...
359,2025.0,craig_doty,1411.793198,1223.0
360,2025.0,paul_corsaro,1404.101904,1237.0
361,2025.0,cornelius_jackson,1613.177099,1267.0
362,2025.0,ethan_faulkner,1608.455994,1464.0


In [6]:
df = elo_teams.merge(elo_coaches, how='left', on='TeamID')
df

,TeamID,TeamELO,Season,CoachName,CoachELO
0,1228,1976.446582,2025.0,brad_underwood,2029.556179
1,1328,1865.209005,2025.0,porter_moser,1918.327714
2,1106,1235.938982,2025.0,tony_madlock,1482.278882
3,1354,1274.504728,2025.0,erik_martin,1520.978014
4,1112,1992.047421,2025.0,tommy_lloyd,2036.358383
...,...,...,...,...,...
744,1478,1155.979444,2025.0,nate_champion,1291.677335
745,1479,1341.454910,2025.0,gary_manchel,1448.532417
746,1480,1258.946093,2025.0,dave_moore,1342.672013
747,3480,1351.409092,NaN,NaN,NaN


In [7]:
#drop rows if there is no Elo for team or coach and TeamID is lower than 3000
df = df[~((df['TeamID'] < 3000) & (df[['CoachELO']].isna().any(axis=1)))]
df

,TeamID,TeamELO,Season,CoachName,CoachELO
0,1228,1976.446582,2025.0,brad_underwood,2029.556179
1,1328,1865.209005,2025.0,porter_moser,1918.327714
2,1106,1235.938982,2025.0,tony_madlock,1482.278882
3,1354,1274.504728,2025.0,erik_martin,1520.978014
4,1112,1992.047421,2025.0,tommy_lloyd,2036.358383
...,...,...,...,...,...
744,1478,1155.979444,2025.0,nate_champion,1291.677335
745,1479,1341.454910,2025.0,gary_manchel,1448.532417
746,1480,1258.946093,2025.0,dave_moore,1342.672013
747,3480,1351.409092,NaN,NaN,NaN


In [8]:
df = df.drop(columns=['CoachName'])
df = df.sort_values(by=['TeamID'])

In [9]:
only_men_teams = df[df['TeamID'] < 3000]
only_women_teams = df[df['TeamID'] >= 3000]
print(only_women_teams.head())
print(only_men_teams.head())

     TeamID      TeamELO  Season  CoachELO
715    3101  1433.625789     NaN       NaN
517    3102  1449.776905     NaN       NaN
331    3103  1307.550808     NaN       NaN
314    3104  2004.344334     NaN       NaN
628    3105  1303.310013     NaN       NaN
     TeamID      TeamELO  Season     CoachELO
712    1101  1459.186427  2025.0  1557.695414
55     1102  1285.078651  2025.0  1388.153462
220    1103  1727.740773  2025.0  1856.454628
103    1104  2069.064662  2025.0  2129.457038
637    1105  1025.906452  2025.0  1271.731597


# Predictions based on Elo, if both teams have coach Elo do predictions based on coach Elo and team elo and mean of both, otherwise do predictions based on team Elo
firstly do the matchups dataframe

In [10]:
df = pd.DataFrame(only_men_teams)

# Tworzenie wszystkich unikalnych kombinacji drużyn
matches = []
for team1, team2 in combinations(df.itertuples(index=False), 2):
    if team1.TeamID < team2.TeamID:
        matches.append((team1.TeamID, team1.TeamELO, team1.CoachELO,
                        team2.TeamID, team2.TeamELO, team2.CoachELO, team1.Season))
    else:
        matches.append((team2.TeamID, team2.TeamELO, team2.CoachELO,
                        team1.TeamID, team1.TeamELO, team1.CoachELO, team1.Season))

# Tworzenie nowej ramki danych
columns = ['Team1_ID', 'Team1_ELO', 'Team1_CoachELO',
           'Team2_ID', 'Team2_ELO', 'Team2_CoachELO', 'Season']
df_matches = pd.DataFrame(matches, columns=columns)

print(df_matches)

       Team1_ID    Team1_ELO  Team1_CoachELO  Team2_ID    Team2_ELO  \
0          1101  1459.186427     1557.695414      1102  1285.078651   
1          1101  1459.186427     1557.695414      1103  1727.740773   
2          1101  1459.186427     1557.695414      1104  2069.064662   
3          1101  1459.186427     1557.695414      1105  1025.906452   
4          1101  1459.186427     1557.695414      1106  1235.938982   
...         ...          ...             ...       ...          ...   
66061      1477  1099.501870     1292.013185      1479  1341.454910   
66062      1477  1099.501870     1292.013185      1480  1258.946093   
66063      1478  1155.979444     1291.677335      1479  1341.454910   
66064      1478  1155.979444     1291.677335      1480  1258.946093   
66065      1479  1341.454910     1448.532417      1480  1258.946093   

       Team2_CoachELO  Season  
0         1388.153462  2025.0  
1         1856.454628  2025.0  
2         2129.457038  2025.0  
3         1271.7315

In [11]:
#check if Team1_ID is lower than Team2_ID
(df_matches['Team1_ID'] < df_matches['Team2_ID']).all()

True

In [12]:
df_matches['Pred_team'] = df_matches.apply(lambda x: predict_proba(x['Team1_ELO'], x['Team2_ELO']), axis=1)
df_matches['Pred_coach'] = df_matches.apply(lambda x: predict_proba(x['Team1_CoachELO'], x['Team2_CoachELO']), axis=1)
df_matches['Pred'] = (df_matches['Pred_team'] + df_matches['Pred_coach']) / 2
df_matches

,Team1_ID,Team1_ELO,Team1_CoachELO,Team2_ID,Team2_ELO,Team2_CoachELO,Season,Pred_team,Pred_coach,Pred
0,1101,1459.186427,1557.695414,1102,1285.078651,1388.153462,2025.0,0.731500,0.726306,0.728903
1,1101,1459.186427,1557.695414,1103,1727.740773,1856.454628,2025.0,0.175676,0.151897,0.163787
2,1101,1459.186427,1557.695414,1104,2069.064662,2129.457038,2025.0,0.029008,0.035870,0.032439
3,1101,1459.186427,1557.695414,1105,1025.906452,1271.731597,2025.0,0.923731,0.838369,0.881050
4,1101,1459.186427,1557.695414,1106,1235.938982,1482.278882,2025.0,0.783319,0.606860,0.695090
...,...,...,...,...,...,...,...,...,...,...
66061,1477,1099.501870,1292.013185,1479,1341.454910,1448.532417,2025.0,0.198962,0.288846,0.243904
66062,1477,1099.501870,1292.013185,1480,1258.946093,1342.672013,2025.0,0.285399,0.427608,0.356504
66063,1478,1155.979444,1291.677335,1479,1341.454910,1448.532417,2025.0,0.255844,0.288449,0.272146
66064,1478,1155.979444,1291.677335,1480,1258.946093,1342.672013,2025.0,0.356010,0.427135,0.391573


In [13]:
#if pred is Nan then Pred=Pred_team
df_matches['Pred'] = df_matches['Pred'].fillna(df_matches['Pred_team'])
df_matches

,Team1_ID,Team1_ELO,Team1_CoachELO,Team2_ID,Team2_ELO,Team2_CoachELO,Season,Pred_team,Pred_coach,Pred
0,1101,1459.186427,1557.695414,1102,1285.078651,1388.153462,2025.0,0.731500,0.726306,0.728903
1,1101,1459.186427,1557.695414,1103,1727.740773,1856.454628,2025.0,0.175676,0.151897,0.163787
2,1101,1459.186427,1557.695414,1104,2069.064662,2129.457038,2025.0,0.029008,0.035870,0.032439
3,1101,1459.186427,1557.695414,1105,1025.906452,1271.731597,2025.0,0.923731,0.838369,0.881050
4,1101,1459.186427,1557.695414,1106,1235.938982,1482.278882,2025.0,0.783319,0.606860,0.695090
...,...,...,...,...,...,...,...,...,...,...
66061,1477,1099.501870,1292.013185,1479,1341.454910,1448.532417,2025.0,0.198962,0.288846,0.243904
66062,1477,1099.501870,1292.013185,1480,1258.946093,1342.672013,2025.0,0.285399,0.427608,0.356504
66063,1478,1155.979444,1291.677335,1479,1341.454910,1448.532417,2025.0,0.255844,0.288449,0.272146
66064,1478,1155.979444,1291.677335,1480,1258.946093,1342.672013,2025.0,0.356010,0.427135,0.391573


In [14]:
#submission df is column with ID = 2025_{Team1_ID}_{Team2_ID} and column with Pred = Pred
df_matches['ID'] = '2025_' + df_matches['Team1_ID'].astype(str) + '_' + df_matches['Team2_ID'].astype(str)
submission = df_matches[['ID', 'Pred']]
submission.index = submission['ID']
submission = submission.drop(columns=['ID'])
submission

,Pred
ID,
2025_1101_1102,0.728903
2025_1101_1103,0.163787
2025_1101_1104,0.032439
2025_1101_1105,0.881050
2025_1101_1106,0.695090
...,...
2025_1477_1479,0.243904
2025_1477_1480,0.356504
2025_1478_1479,0.272146


In [15]:
df = pd.DataFrame(only_women_teams)

# Tworzenie wszystkich unikalnych kombinacji drużyn
matches = []
for team1, team2 in combinations(df.itertuples(index=False), 2):
    if team1.TeamID < team2.TeamID:
        matches.append((team1.TeamID, team1.TeamELO, team1.CoachELO,
                        team2.TeamID, team2.TeamELO, team2.CoachELO, team1.Season))
    else:
        matches.append((team2.TeamID, team2.TeamELO, team2.CoachELO,
                        team1.TeamID, team1.TeamELO, team1.CoachELO, team1.Season))

# Tworzenie nowej ramki danych
columns = ['Team1_ID', 'Team1_ELO', 'Team1_CoachELO',
           'Team2_ID', 'Team2_ELO', 'Team2_CoachELO', 'Season']
df_matches = pd.DataFrame(matches, columns=columns)

print(df_matches)

       Team1_ID    Team1_ELO  Team1_CoachELO  Team2_ID    Team2_ELO  \
0          3101  1433.625789             NaN      3102  1449.776905   
1          3101  1433.625789             NaN      3103  1307.550808   
2          3101  1433.625789             NaN      3104  2004.344334   
3          3101  1433.625789             NaN      3105  1303.310013   
4          3101  1433.625789             NaN      3106   967.118632   
...         ...          ...             ...       ...          ...   
67891      3477  1113.947230             NaN      3479  1190.234158   
67892      3477  1113.947230             NaN      3480  1351.409092   
67893      3478  1178.688631             NaN      3479  1190.234158   
67894      3478  1178.688631             NaN      3480  1351.409092   
67895      3479  1190.234158             NaN      3480  1351.409092   

       Team2_CoachELO  Season  
0                 NaN     NaN  
1                 NaN     NaN  
2                 NaN     NaN  
3                 N

In [16]:
df_matches=df_matches.drop(columns=['Season', 'Team1_CoachELO', 'Team2_CoachELO'])
df_matches

,Team1_ID,Team1_ELO,Team2_ID,Team2_ELO
0,3101,1433.625789,3102,1449.776905
1,3101,1433.625789,3103,1307.550808
2,3101,1433.625789,3104,2004.344334
3,3101,1433.625789,3105,1303.310013
4,3101,1433.625789,3106,967.118632
...,...,...,...,...
67891,3477,1113.947230,3479,1190.234158
67892,3477,1113.947230,3480,1351.409092
67893,3478,1178.688631,3479,1190.234158
67894,3478,1178.688631,3480,1351.409092


In [17]:
df_matches['Pred'] = df_matches.apply(lambda x: predict_proba(x['Team1_ELO'], x['Team2_ELO']), axis=1)
df_matches

,Team1_ID,Team1_ELO,Team2_ID,Team2_ELO,Pred
0,3101,1433.625789,3102,1449.776905,0.476773
1,3101,1433.625789,3103,1307.550808,0.673871
2,3101,1433.625789,3104,2004.344334,0.036078
3,3101,1433.625789,3105,1303.310013,0.679213
4,3101,1433.625789,3106,967.118632,0.936161
...,...,...,...,...,...
67891,3477,1113.947230,3479,1190.234158,0.391945
67892,3477,1113.947230,3480,1351.409092,0.203115
67893,3478,1178.688631,3479,1190.234158,0.483391
67894,3478,1178.688631,3480,1351.409092,0.270072


In [18]:
df_matches['ID'] = '2025_' + df_matches['Team1_ID'].astype(str) + '_' + df_matches['Team2_ID'].astype(str)
submission_women = df_matches[['ID', 'Pred']]
submission_women.index = submission_women['ID']
submission_women= submission_women.drop(columns=['ID'])
submission_women

,Pred
ID,
2025_3101_3102,0.476773
2025_3101_3103,0.673871
2025_3101_3104,0.036078
2025_3101_3105,0.679213
2025_3101_3106,0.936161
...,...
2025_3477_3479,0.391945
2025_3477_3480,0.203115
2025_3478_3479,0.483391


In [19]:
#submission is submission and submission_women
submission = pd.concat([submission, submission_women])
submission

,Pred
ID,
2025_1101_1102,0.728903
2025_1101_1103,0.163787
2025_1101_1104,0.032439
2025_1101_1105,0.881050
2025_1101_1106,0.695090
...,...
2025_3477_3479,0.391945
2025_3477_3480,0.203115
2025_3478_3479,0.483391


In [22]:
sample_submission = pd.read_csv('SampleSubmissionStage2.csv')
sample_submission = sample_submission.drop(columns=['Pred'])
sample_submission

,ID
0,2025_1101_1102
1,2025_1101_1103
2,2025_1101_1104
3,2025_1101_1105
4,2025_1101_1106
...,...
131402,2025_3477_3479
131403,2025_3477_3480
131404,2025_3478_3479
131405,2025_3478_3480


In [23]:
submission = sample_submission.merge(submission, how='left', on='ID')
submission.index = submission['ID']
submission = submission.drop(columns=['ID'])
submission

,Pred
ID,
2025_1101_1102,0.728903
2025_1101_1103,0.163787
2025_1101_1104,0.032439
2025_1101_1105,0.881050
2025_1101_1106,0.695090
...,...
2025_3477_3479,0.391945
2025_3477_3480,0.203115
2025_3478_3479,0.483391


In [25]:
submission.to_csv('only_elo_submission_corrected.csv')